# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Explore record sets and their fields using the Croissant metadata
print("Available Record Sets:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()
    # Show first record as an example
    records_iter = dataset.records(record_set=rs.id)
    try:
        first_rec = next(records_iter)
        print(f"  Example record: {first_rec}")
    except StopIteration:
        print("  No records found.")
    print("-"*40)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract all available record sets into DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# For illustration, select the first record set for further steps
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None

if main_record_set_id:
    print(f"Loaded columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# For demonstration, check for numeric columns and apply EDA routines
import numpy as np
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Attempt to find a numeric field by data type or by sniffing columns
    # We'll use the field metadata if possible
    numeric_field_id = None
    group_field_id = None

    # Search for a numeric column
    for rs in dataset.metadata.record_sets:
        if rs.id == main_record_set_id:
            for field in rs.fields:
                if field.data_type in ('schema:Float', 'schema:Number', 'schema:Integer', 'Float', 'Number', 'Integer'):
                    if field.id in df.columns:
                        numeric_field_id = field.id
                        break
            for field in rs.fields:
                if field.data_type == 'schema:Text' or field.data_type == 'Text':
                    # Use as group field if not numeric
                    if field.id in df.columns:
                        group_field_id = field.id
                        break

    if numeric_field_id:
        # Try to convert the numeric field to float just in case
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanmean(df[numeric_field_id]) if np.isfinite(np.nanmean(df[numeric_field_id])) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (mean value):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped average of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for this record set. Consider updating the notebook to use the correct field @id.")
else:
    print("No main record set loaded, cannot proceed with EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Scatter/group plot if group is available and categorical
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=90)
        plt.show()
else:
    print("Visualization skipped: numeric field or main record set not present.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using the `mlcroissant` library, we loaded the Croissant metadata and records for the FAIR² dataset.
- We identified available record sets and their fields via their `@id`s.
- Explored a sample record set, filtered by a key numeric field, performed normalization, grouping, and visualized the data distribution.
- This workflow can be adapted for more specialized analysis by selecting specific fields or combining relationships across record sets, always referencing them by their `@id` for clarity and reproducibility.